<a href="https://colab.research.google.com/github/alvillegasru/15_kV_Arc_Flash/blob/main/Cuadernos/Analisis_Comparativo_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Análisis Comparativo entre metodologías de cálculo de energía incidente aplicables a tensiones superiores a 15 kV utilizando métodos estadísticos y de Machine Learning

In [ ]:
#@title Clonación del repositorio
!git clone https://github.com/alvillegasru/15_kV_Arc_Flash.git # Copiar el repositorio de GitHub del curso

In [ ]:
#@title Instalación de librerías
import os

# Define la ruta al archivo txt
ruta_requirements = os.path.join('15_kV_Arc_Flash', 'Archivos', 'Datos_de_entrada', 'Data_Analisis_Comparativo', 'Requirements_Analisis.txt')

# Instala las librerías con pip
!pip install -r {ruta_requirements}

In [ ]:
#@title Importación de librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import root_mean_squared_error, r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from ipywidgets import Checkbox, FloatRangeSlider, IntRangeSlider, Dropdown, Button, Output, VBox, HBox, interact, FloatSlider
import joblib
from IPython.display import display
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasRegressor

In [ ]:
#@title Verificación de versiones de librerías instaladas
for libreria in ['pandas', 'numpy', 'matplotlib', 'openpyxl', 'seaborn', 'sklearn', 'ipywidgets', 'joblib', 'IPython', 'tensorflow', 'scikeras']:
    try:
        # Importar dinámicamente la librería
        modulo = __import__(libreria)
        # Mostrar la versión
        print(f"{libreria}: {modulo.__version__}")
    except ImportError:
        # Mostrar mensaje si la librería no está instalada
        print(f"{libreria}: No instalada")

In [ ]:
#@title Lectura de datasets de entrenamiento y validación

#Consideraciones:
#La creación de estos datasets se realizaron en el cuaderno Creacion_Datasets.ipynb que se encuentra en la siguiente ruta del repositorio: 15_kV_Arc_Flash/Cuadernos/Creacion_Datasets.ipynb

#Ubicación de los datasets
file_path_entrenamiento = "15_kV_Arc_Flash/Archivos/Datos_de_entrada/Data_Analisis_Comparativo/dataset_entrenamiento.xlsx"

file_path_validacion = "15_kV_Arc_Flash/Archivos/Datos_de_entrada/Data_Analisis_Comparativo/dataset_validacion.xlsx"

#Lectura de datasets
dataset_entrenamiento = pd.read_excel(file_path_entrenamiento)
dataset_validacion = pd.read_excel(file_path_validacion)

In [ ]:
#@title Análisis Exploratorio de Datos

def explore_data(info_type, df_name):
    """
    Explora los datos del dataset de forma interactiva.

    Permite visualizar diferentes aspectos del dataset, como el encabezado,
    la descripción, la información general, los valores nulos y la distribución
    de los atributos.

    Parámetros:
        info_type (str): Tipo de información a mostrar. Puede ser:
            - 'Encabezado': Muestra las primeras filas del dataset.
            - 'Descripción': Muestra estadísticas descriptivas del dataset.
            - 'Información': Muestra información general sobre el dataset.
            - 'Valores Nulos': Muestra la cantidad de valores nulos por columna.
            - 'Distribución de Atributos': Muestra histogramas de los atributos numéricos.
        df_name (str): El nombre del DataFrame a explorar. Puede ser 'Dataset Entrenamiento' o 'Dataset Validación'.

    Retorna:
        None. La función muestra la información solicitada en la salida.
    """
    # Get the DataFrame based on df_name
    if df_name == 'Dataset Entrenamiento':
        df = dataset_entrenamiento
    elif df_name == 'Dataset Validación':
        df = dataset_validacion
    else:
        print("Nombre de DataFrame no válido.")
        return

    if info_type == 'Encabezado':
        display(df.head())
    elif info_type == 'Descripción':
        display(df.describe())
    elif info_type == 'Información':
        display(df.info())
    elif info_type == 'Valores Nulos':
        display(df.isnull().sum())
    elif info_type == 'Distribución de Atributos':
        df.hist(bins=50, figsize=(20, 15))
        plt.show()
    else:
        print("Opción no válida.")

# Menú interactivo para seleccionar la información y el DataFrame que se desea explorar
interact(explore_data,
         info_type=Dropdown(options=['Encabezado', 'Descripción', 'Información', 'Valores Nulos', 'Distribución de Atributos'],
                            value='Encabezado', description='Datos:'),
         df_name=Dropdown(options=['Dataset Entrenamiento', 'Dataset Validación'],
                         value='Dataset Entrenamiento', description='DataFrame:'));


In [ ]:
#@title Limpieza de datos
#Considerando los valores NaN generados por resultados de la metodología de cálculo EPRI para gaps pequeños, se considera necesario eliminar las filas con gaps menores a 200 mm, con el fin de evitar ruido en el análisis.

# Eliminar filas con valores filas con gap menor a 200 mm
dataset_entrenamiento = dataset_entrenamiento[dataset_entrenamiento['Gap (G) [mm]'] >= 200]
dataset_validacion = dataset_validacion[dataset_validacion['Gap (G) [mm]'] >= 200]

# Restaurar el índice
dataset_entrenamiento = dataset_entrenamiento.reset_index(drop=True)
dataset_validacion = dataset_validacion.reset_index(drop=True)

In [ ]:
#@title Impacto de variables de entrada en metodología IEEE Std- 1584 – 2018

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'
os.makedirs(dir_path, exist_ok=True)  # Crear la carpeta si no existe

# Definir las configuraciones de electrodos
configuraciones = ['HCB', 'VCB', 'VCBB']

# Seleccionar las columnas de interés para el eje X
columnas_x = [dataset_entrenamiento.columns[1], dataset_entrenamiento.columns[2], dataset_entrenamiento.columns[4], dataset_entrenamiento.columns[5]]

# Iterar sobre cada columna en X y generar gráficos
for col_x in columnas_x:
    fig, axes = plt.subplots(3, 1, figsize=(15, 30))  # 3 filas, 1 columna

    for i, config in enumerate(configuraciones):
        # Filtrar los datos para la configuración actual
        data_config = dataset_entrenamiento[dataset_entrenamiento['Configuración de electrodos (EC)'] == config]

        # Crear la gráfica
        sns.scatterplot(x=col_x, y='IEEE 1584 - 2018 Earc [cal/cm^2]',
                        hue='Caso de estudio', data=data_config, ax=axes[i], palette='viridis')
        axes[i].set_title(f'IEEE 1584 - 2018 Earc vs {col_x} para EC = {config}', fontsize=18, fontweight='bold')
        axes[i].set_xlabel(col_x, fontsize=16)
        axes[i].set_ylabel('IEEE 1584 - 2018 Earc [cal/cm^2]', fontsize=16)
        axes[i].grid(True)
        legend = axes[i].legend(title='Caso de estudio', loc='upper right', fontsize=12)
        legend.get_title().set_fontsize('12')  # Cambiar el tamaño del título de la leyenda

        # Guardar la imagen del subplot actual
        extent = axes[i].get_window_extent().transformed(fig.dpi_scale_trans.inverted())
        file_path = os.path.join(dir_path, f'Grafico_IEEE_{col_x}_{config}.png')
        fig.savefig(file_path, bbox_inches=extent.expanded(1.15, 1.2))  # Expandir el bbox para incluir la leyenda

    # Mostrar la figura con las tres gráficas (opcional)
    plt.show()

In [ ]:
#@title Impacto de variables de entrada en metodología EPRI

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'
os.makedirs(dir_path, exist_ok=True)  # Crear la carpeta si no existe

# Seleccionar las columnas de interés para el eje X
columnas_x = [dataset_entrenamiento.columns[1], dataset_entrenamiento.columns[2], dataset_entrenamiento.columns[4], dataset_entrenamiento.columns[5]]

# Iterar sobre cada columna en X y generar gráficos
for col_x in columnas_x:
    # Filtrar los datos para la configuración actual
    data_config = dataset_entrenamiento[dataset_entrenamiento['Configuración de electrodos (EC)'] == 'HCB']  # Usar configuraciones[0] para HCB

    # Crear la gráfica
    plt.figure(figsize=(15, 10))  # Crear una nueva figura para cada gráfica
    sns.scatterplot(x=col_x, y='EPRI Earc Open Air [cal/cm^2]',
                    hue='Caso de estudio', data=data_config, palette='viridis')
    plt.title(f'EPRI Earc Open Air vs {col_x}', fontsize=18, fontweight='bold')
    plt.xlabel(col_x, fontsize=16)
    plt.ylabel('EPRI Earc Open Air [cal/cm^2]', fontsize=16)
    plt.grid(True)
    legend = plt.legend(title='Caso de estudio', loc='upper right', fontsize=12)
    legend.get_title().set_fontsize('12')  # Cambiar el tamaño del título de la leyenda

    # Guardar la imagen
    file_path = os.path.join(dir_path, f'Grafico_EPRI_{col_x}.png')  # Usar configuraciones[0] para HCB
    plt.savefig(file_path, bbox_inches='tight')
    plt.show()
    plt.close()  # Cerrar la figura para liberar memoria



In [ ]:
#@title Impacto de variables de entrada en metodología Terzija/Koglin

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'
os.makedirs(dir_path, exist_ok=True)  # Crear la carpeta si no existe

# Seleccionar las columnas de interés para el eje X
columnas_x = [dataset_entrenamiento.columns[1], dataset_entrenamiento.columns[2], dataset_entrenamiento.columns[4], dataset_entrenamiento.columns[5]]

# Iterar sobre cada columna en X y generar gráficos
for col_x in columnas_x:
    # Filtrar los datos para la configuración actual
    data_config = dataset_entrenamiento[dataset_entrenamiento['Configuración de electrodos (EC)'] == 'HCB']  # Usar configuraciones[0] para HCB

    # Crear la gráfica
    plt.figure(figsize=(15, 10))  # Crear una nueva figura para cada gráfica
    sns.scatterplot(x=col_x, y='Terzija/Koglin Earc Open Air [cal/cm^2]',
                    hue='Caso de estudio', data=data_config, palette='viridis')
    plt.title(f'Terzija/Koglin Earc Open Air vs {col_x}', fontsize=18, fontweight='bold')
    plt.xlabel(col_x, fontsize=16)
    plt.ylabel('Terzija/Koglin Earc Open Air [cal/cm^2]', fontsize=16)
    plt.grid(True)
    legend = plt.legend(title='Caso de estudio', loc='upper right', fontsize=12)
    legend.get_title().set_fontsize('12')  # Cambiar el tamaño del título de la leyenda

    # Guardar la imagen
    file_path = os.path.join(dir_path, f'Grafico_Terzija_{col_x}.png')  # Usar configuraciones[0] para HCB
    plt.savefig(file_path, bbox_inches='tight')
    plt.show()
    plt.close()  # Cerrar la figura para liberar memoria



In [ ]:
#@title Impacto de variables de entrada en Relación IEEE - EPRI Earc Open Air

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'
os.makedirs(dir_path, exist_ok=True)  # Crear la carpeta si no existe

# Definir las configuraciones de electrodos
configuraciones = ['HCB', 'VCB', 'VCBB']

# Seleccionar las columnas de interés para el eje X
columnas_x = [dataset_entrenamiento.columns[1], dataset_entrenamiento.columns[2], dataset_entrenamiento.columns[4], dataset_entrenamiento.columns[5]]

# Iterar sobre cada columna en X y generar gráficos
for col_x in columnas_x:
    fig, axes = plt.subplots(3, 1, figsize=(15, 30))  # 3 filas, 1 columna

    for i, config in enumerate(configuraciones):
        # Filtrar los datos para la configuración actual
        data_config = dataset_entrenamiento[dataset_entrenamiento['Configuración de electrodos (EC)'] == config]

        # Crear la gráfica
        sns.scatterplot(x=col_x, y='Relación IEEE - EPRI Earc Open Air',
                        hue='Caso de estudio', data=data_config, ax=axes[i], palette='viridis')
        axes[i].set_title(f'Relación IEEE - EPRI vs {col_x} para EC = {config}', fontsize=18, fontweight='bold')
        axes[i].set_xlabel(col_x, fontsize=16)
        axes[i].set_ylabel('Relación IEEE - EPRI Earc Open Air', fontsize=16)
        axes[i].grid(True)
        legend = axes[i].legend(title='Caso de estudio', loc='upper right', fontsize=12)
        legend.get_title().set_fontsize('12')  # Cambiar el tamaño del título de la leyenda

        # Guardar la imagen del subplot actual
        extent = axes[i].get_window_extent().transformed(fig.dpi_scale_trans.inverted())
        file_path = os.path.join(dir_path, f'Grafico_Rel_IEEE_EPRI_{col_x}_{config}.png')
        fig.savefig(file_path, bbox_inches=extent.expanded(1.15, 1.2))  # Expandir el bbox para incluir la leyenda

    # Mostrar la figura con las tres gráficas (opcional)
    plt.show()

In [ ]:
#@title Impacto de variables de entrada en Relación IEEE - Terzija Earc Open Air

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'
os.makedirs(dir_path, exist_ok=True)  # Crear la carpeta si no existe

# Definir las configuraciones de electrodos
configuraciones = ['HCB', 'VCB', 'VCBB']

# Seleccionar las columnas de interés para el eje X
columnas_x = [dataset_entrenamiento.columns[1], dataset_entrenamiento.columns[2], dataset_entrenamiento.columns[4], dataset_entrenamiento.columns[5]]

# Iterar sobre cada columna en X y generar gráficos
for col_x in columnas_x:
    fig, axes = plt.subplots(3, 1, figsize=(15, 30))  # 3 filas, 1 columna

    for i, config in enumerate(configuraciones):
        # Filtrar los datos para la configuración actual
        data_config = dataset_entrenamiento[dataset_entrenamiento['Configuración de electrodos (EC)'] == config]

        # Crear la gráfica
        sns.scatterplot(x=col_x, y='Relación IEEE - Terzija Earc Open Air',
                        hue='Caso de estudio', data=data_config, ax=axes[i], palette='viridis')
        axes[i].set_title(f'Relación IEEE - Terzija vs {col_x} para EC = {config}', fontsize=18, fontweight='bold')
        axes[i].set_xlabel(col_x, fontsize=16)
        axes[i].set_ylabel('Relación IEEE - Terzija Earc Open Air', fontsize=16)
        axes[i].grid(True)
        legend = axes[i].legend(title='Caso de estudio', loc='upper right', fontsize=12)
        legend.get_title().set_fontsize('12')  # Cambiar el tamaño del título de la leyenda

        # Guardar la imagen del subplot actual
        extent = axes[i].get_window_extent().transformed(fig.dpi_scale_trans.inverted())
        file_path = os.path.join(dir_path, f'Grafico_Rel_IEEE_Terzija_{col_x}_{config}.png')
        fig.savefig(file_path, bbox_inches=extent.expanded(1.15, 1.2))  # Expandir el bbox para incluir la leyenda

    # Mostrar la figura con las tres gráficas (opcional)
    plt.show()

In [ ]:
#@title Variables de interés de análisis de Relación IEEE - EPRI/Terzija Earc
#Las variables con mayor impacto en la predicción de nuevos valores de 'Relación IEEE - EPRI Earc Open Air' y 'Relación IEEE - Terzija Earc Open Air' son: 'Tensión nominal (V) [kV]', 'Corriente de cortocircuito (Ibf) [kA]', 'Distancia de Trabajo (D) [mm]', 'Gap (G) [mm]', 'Configuración de electrodos (EC)'
#El resto de variables son dependientes de las variables de interés o no hacen parte de las ecuaciones de cálculo planteadas en cada una de las metolodgías de cálculo, por ende, se descartan para la creación y entrenemiento de los modelos a usar

columnas_de_interes = ['Tensión nominal (V) [kV]', 'Corriente de cortocircuito (Ibf) [kA]', 'Distancia de Trabajo (D) [mm]', 'Gap (G) [mm]', 'Configuración de electrodos (EC)', 'Relación IEEE - EPRI Earc Open Air', 'Relación IEEE - Terzija Earc Open Air']

dataset_entrenamiento = dataset_entrenamiento[columnas_de_interes]
dataset_validacion = dataset_validacion[columnas_de_interes]

In [ ]:
#@title Codificación de variables categóricas
# Crear una instancia de OrdinalEncoder
encoder = OrdinalEncoder()
categorical_cols_entrenamiento = dataset_entrenamiento.select_dtypes(include=['object']).columns  # Columnas categóricas dataset entrenamiento
categorical_cols_validacion = dataset_validacion.select_dtypes(include=['object']).columns  # Columnas categóricas dataset validación

# Ajustar y transformar las columnas categóricas del dataset de entrenamiento
dataset_entrenamiento[categorical_cols_entrenamiento] = encoder.fit_transform(dataset_entrenamiento[categorical_cols_entrenamiento])

# Ajustar y transformar las columnas categóricas del dataset de validación
dataset_validacion[categorical_cols_validacion] = encoder.fit_transform(dataset_validacion[categorical_cols_validacion])

print("Cambios en dataset entrenamiento\n")
for i, categoria in enumerate(encoder.categories_):
    print(f"Variable: {dataset_entrenamiento.columns[i]}")
    for j, valor in enumerate(categoria):
        print(f"  {valor} -> {j}")
    print("\n")

print("Cambios en dataset validación\n")
for i, categoria in enumerate(encoder.categories_):
    print(f"Variable: {dataset_validacion.columns[i]}")
    for j, valor in enumerate(categoria):
        print(f"  {valor} -> {j}")
    print("\n")

In [ ]:
#@title Correlación entre columnas numéricas para el dataset entrenamiento y validación
# Ordenamiento de los datos para generar un visualización adecuada de las correlaciones entre variables

def tidy_corr_matrix(corr_mat):
    """
    Función para convertir una matriz de correlación de pandas en formato tidy.

    Parámetros:
        corr_mat (pd.DataFrame): Matriz de correlación de pandas.

    Retorna:
        pd.DataFrame: Matriz de correlación en formato tidy,
                      con columnas 'variable_1', 'variable_2', 'r' y 'abs_r',
                      ordenada por 'abs_r' de forma descendente.
    """
    corr_mat = corr_mat.stack().reset_index()
    corr_mat.columns = ['variable_1','variable_2','r']
    corr_mat = corr_mat.loc[corr_mat['variable_1'] != corr_mat['variable_2'], :]
    corr_mat['abs_r'] = np.abs(corr_mat['r'])
    corr_mat = corr_mat.sort_values('abs_r', ascending=False)

    return(corr_mat)

In [ ]:
corr_matrix_entrenamiento = dataset_entrenamiento.select_dtypes(include=['float64', 'int']) \
              .corr(method='pearson')
display(tidy_corr_matrix(corr_matrix_entrenamiento))

In [ ]:
corr_matrix_validacion = dataset_validacion.select_dtypes(include=['float64', 'int']) \
              .corr(method='pearson')
display(tidy_corr_matrix(corr_matrix_validacion))

In [ ]:
#@title Heatmap matriz de correlaciones - dataset entrenamiento

fig, ax_entrenamiento = plt.subplots(nrows=1, ncols=1, figsize=(10, 9))

sns.heatmap(
    corr_matrix_entrenamiento, annot=True, fmt=".2f", linewidths=.5,
    cmap='viridis', ax = ax_entrenamiento, vmin=-1, vmax=1
)

ax_entrenamiento.tick_params(labelsize = 10)
file_path = os.path.join(dir_path, f'Mapa_calor_entrenamiento.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura

In [ ]:
#@title Heatmap matriz de correlaciones - dataset validación

fig, ax_validacion = plt.subplots(nrows=1, ncols=1, figsize=(10, 9))

sns.heatmap(
    corr_matrix_validacion, annot=True, fmt=".2f", linewidths=.5, cmap='viridis', ax = ax_validacion, vmin=-1, vmax=1
)

ax_validacion.tick_params(labelsize = 10)
file_path = os.path.join(dir_path, f'Mapa_calor_validacion.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura

In [ ]:
#@title Dataframes metodologías Terzija/Koglin y EPRI
df_entrenamiento_terzija = dataset_entrenamiento.drop(columns='Relación IEEE - EPRI Earc Open Air')
df_entrenamiento_epri = dataset_entrenamiento.drop(columns='Relación IEEE - Terzija Earc Open Air')

In [ ]:
df_entrenamiento_terzija.info()

In [ ]:
df_entrenamiento_epri.info()

##Análisis de modelos de ML para predicción de las variables 'Relación IEEE - EPRI Earc Open Air' y 'Relación IEEE - Terzija Open Air'

In [ ]:
#@title División de datos entrada y salida

#Datos entrada Terzija/Koglin
x_terz = df_entrenamiento_terzija.drop(columns='Relación IEEE - Terzija Earc Open Air')
#Datos salida Terzija/Koglin
y_terz = df_entrenamiento_terzija['Relación IEEE - Terzija Earc Open Air']

#Datos entrada EPRI
x_epri = df_entrenamiento_epri.drop(columns='Relación IEEE - EPRI Earc Open Air')
#Datos salida EPRI
y_epri = df_entrenamiento_epri['Relación IEEE - EPRI Earc Open Air']

In [ ]:
#@title Funciones para creación de modelos

def conjuntos_de_datos(x, y):
    """
    Divide los datos en conjuntos de entrenamiento, prueba y validación.

    Esta función toma los datos de entrada (X) y salida (y) y los divide en tres conjuntos:
    entrenamiento y prueba. El conjunto de entrenamiento se utiliza para entrenar el modelo y el conjunto de prueba se utiliza para evaluar el rendimiento del modelo durante el entrenamiento.

    Parámetros:
        x (pd.DataFrame): Los datos de entrada.
        y (pd.Series): Los datos de salida.

    Retorna:
        Una tupla que contiene los siguientes conjuntos de datos:
            - x_train: Datos de entrada para entrenamiento.
            - x_test: Datos de entrada para prueba.
            - y_train: Datos de salida para entrenamiento.
            - y_test: Datos de salida para prueba.
    """
    # Dividir los datos en conjunto de entrenamiento-prueba y validación
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
    return x_train, x_test, y_train, y_test

def crear_modelo_polinomial_grado_2(x_train, y_train):
    """
    Crea y entrena un modelo de regresión polinomial de grado 2.

    Esta función crea una instancia de un modelo de regresión polinomial de grado 2 utilizando PolynomialFeatures y LinearRegression de scikit-learn. Luego, entrena el modelo con los datos de entrenamiento proporcionados.

    Parámetros:
        x_train (pd.DataFrame): Datos de entrada para entrenamiento.
        y_train (pd.Series): Datos de salida para entrenamiento.

    Retorna:
        Un objeto LinearRegression entrenado que representa el modelo polinomial.
    """
    # Inicialización de Modelo de regresión polinómica de grado 2
    modelo = make_pipeline(
			PolynomialFeatures(degree=2),
			StandardScaler(),
			LinearRegression()
	)
    # Entrenamiento del modelo de regresión lineal con características polinómicas
    modelo.fit(x_train, y_train)
    return modelo

def crear_modelo_elastic_net(x_train, y_train):
	"""
    Crea y entrena un modelo de regresión Elastic Net.

	Esta función crea una instancia de un modelo de regresión Elastic Net
	utilizando ElasticNet de scikit-learn. Luego, entrena el modelo con los datos de entrenamiento proporcionados.

	Parámetros:
		x_train (pd.DataFrame): Datos de entrada para entrenamiento.
		y_train (pd.Series): Datos de salida para entrenamiento.

	Retorna:
		Un objeto ElasticNet entrenado que representa el modelo.
	"""
	# Inicialización de Modelo de regresión con ElasticNet
	modelo = make_pipeline(
			StandardScaler(),
			ElasticNet(alpha=0.5, l1_ratio=0.1)
	)
	# Entrenamiento del modelo de regresión lineal
	modelo.fit(x_train, y_train)
	return modelo

def crear_red_neuronal(x_train, x_test, y_train, y_test):
	"""
    Crea y entrena una red neuronal.

	Esta función crea una instancia de una red neuronal del tipo Perceptrón Multicapa
	utilizando Keras/TensorFlow. Luego,
	entrena el modelo con los datos de entrenamiento proporcionados.

	Parámetros:
		x_train (pd.DataFrame): Datos de entrada para entrenamiento.
		x_test (pd.DataFrame): Datos de entrada para prueba.
		y_train (pd.Series): Datos de salida para entrenamiento.
		y_test (pd.Series): Datos de salida para prueba.


	Retorna:
		Un objeto Sequential entrenado que representa el modelo de red neuronal.
	"""

	# Crear el modelo
	modelo_inicial = Sequential([
		Dense(64, activation='relu', input_shape=(x_train.shape[1],)),
		Dense(32, activation='relu'),
		Dense(16, activation='relu'),
		Dense(1)  # Una sola salida para regresión
	])

	# Compilar el modelo
	modelo_inicial.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

	modelo = make_pipeline(
    StandardScaler(),
    KerasRegressor(modelo_inicial, epochs=100, batch_size=32, verbose=1)
	)

	# Entrenar el modelo
	modelo.fit(x_train, y_train)

	return modelo

In [ ]:
#@title Función para la evaluación del modelo

def evaluar_modelo(y_train, y_pred_train, y_test, y_pred_test, y_val, y_pred_val):
	"""
    Evalúa el rendimiento de un modelo de regresión.

	Esta función calcula e imprime las métricas de evaluación del modelo
	para los conjuntos de entrenamiento, prueba y validación.

	Parámetros:
		y_train (pd.Series): Valores reales del conjunto de entrenamiento.
		y_pred_train (pd.Series): Valores predichos por el modelo para el conjunto de entrenamiento.
		y_test (pd.Series): Valores reales del conjunto de prueba.
		y_pred_test (pd.Series): Valores predichos por el modelo para el conjunto de prueba.
		y_val (pd.Series): Valores reales del conjunto de validación.
		y_pred_val (pd.Series): Valores predichos por el modelo para el conjunto de validación.

	Retorna:
		tuple: Una tupla que contiene las métricas de evaluación (MAE, RMSE, R²) para cada conjunto de datos (train, test, val).
	"""
	#Evaluación del conjunto de entrenamiento
	mae_train = mean_absolute_error(y_train, y_pred_train)
	rmse_train = root_mean_squared_error(y_train, y_pred_train)
	r2_train = r2_score(y_train, y_pred_train)
	print(f"MAE conjunto entrenamiento: {mae_train:.4f}")
	print(f"RMSE conjunto entrenamiento: {rmse_train:.4f}")
	print(f"R2 conjunto entrenamiento: {r2_train:.4f}")
	print("")

	#Evaluación del conjunto de prueba
	mae_test = mean_absolute_error(y_test, y_pred_test)
	rmse_test = root_mean_squared_error(y_test, y_pred_test)
	r2_test = r2_score(y_test, y_pred_test)
	print(f"MAE conjunto prueba: {mae_test:.4f}")
	print(f"RMSE conjunto prueba: {rmse_test:.4f}")
	print(f"R2 conjunto prueba: {r2_test:.4f}")
	print("")

	#Evaluación del conjunto de validación
	mae_val = mean_absolute_error(y_val, y_pred_val)
	rmse_val = root_mean_squared_error(y_val, y_pred_val)
	r2_val = r2_score(y_val, y_pred_val)
	print(f"MAE conjunto validación: {mae_val:.4f}")
	print(f"RMSE conjunto validación: {rmse_val:.4f}")
	print(f"R2 conjunto validación: {r2_val:.4f}")

	return mae_train, rmse_train, r2_train, mae_test, rmse_test, r2_test, mae_val, rmse_val, r2_val


In [ ]:
#@title Entrenamiento de modelos - Metodología Terzija/Koglin

#Conjunto de datos
x_train_terz, x_test_terz, y_train_terz, y_test_terz = conjuntos_de_datos(x_terz, y_terz)

#Entrenamiento con modelo ElasticNet
modelo_terzija_elastic_net = crear_modelo_elastic_net(x_train_terz, y_train_terz)
#Entrenamiento con modelo Polynomial grade 2
modelo_terzija_polinomial = crear_modelo_polinomial_grado_2(x_train_terz, y_train_terz)
#Entrenamiento con Redes Neuronales del tipo Perceptrón Multicapa
modelo_terzija_red_neuronal = crear_red_neuronal(x_train_terz, x_test_terz, y_train_terz, y_test_terz)

In [ ]:
#@title Entrenamiento de modelos - Metodología EPRI

# Conjunto de datos
x_train_epri, x_test_epri, y_train_epri, y_test_epri = conjuntos_de_datos(x_epri, y_epri)

# Entrenamiento con modelo ElasticNet
modelo_epri_elastic_net = crear_modelo_elastic_net(x_train_epri, y_train_epri)

# Entrenamiento con modelo Polynomial grade 2
modelo_epri_polinomial = crear_modelo_polinomial_grado_2(x_train_epri, y_train_epri)

# Entrenamiento con Redes Neuronales del tipo Perceptrón Multicapa
modelo_epri_red_neuronal = crear_red_neuronal(x_train_epri, x_test_epri, y_train_epri, y_test_epri)

In [ ]:
#@title Media y desviación estándar de los datos
#Se extrae el valor de la media y la desviación estándar para visualizar su diferencia con respecto al resultado RMSE de cada modelo

# Calcular la media y desviación estándar para y_terz
mean_y_terz = y_terz.mean()
std_y_terz = y_terz.std()

print(f"La media de y_terz es: {mean_y_terz}")
print(f"La desviación estándar de y_terz es: {std_y_terz}")
print("")

# Calcular la media y desviación estándar para y_epri
mean_y_epri = y_epri.mean()
std_y_epri = y_epri.std()

print(f"La media de y_epri es: {mean_y_epri}")
print(f"La desviación estándar de y_epri es: {std_y_epri}")

In [ ]:
#@title Dataframes metodologías Terzija/Koglin y EPRI - dataset validación
df_validacion_terzija = dataset_validacion.drop(columns='Relación IEEE - EPRI Earc Open Air')
df_validacion_epri = dataset_validacion.drop(columns='Relación IEEE - Terzija Earc Open Air')

#Datos entrada Terzija/Koglin
x_dataval_terz = df_validacion_terzija.drop(columns='Relación IEEE - Terzija Earc Open Air')
#Datos salida Terzija/Koglin
y_dataval_terz = df_validacion_terzija['Relación IEEE - Terzija Earc Open Air']

#Datos entrada EPRI
x_dataval_epri = df_validacion_epri.drop(columns='Relación IEEE - EPRI Earc Open Air')
#Datos salida EPRI
y_dataval_epri = df_validacion_epri['Relación IEEE - EPRI Earc Open Air']

In [ ]:
#@title Media y desviación estándar de los datos
#Se extrae el valor de la media y la desviación estándar para visualizar su diferencia con respecto al resultado RMSE de cada modelo

# Calcular la media y desviación estándar para y_terz
mean_y_dataval_terz = y_dataval_terz.mean()
std_y_dataval_terz = y_dataval_terz.std()

print(f"La media de y_dataval_terz es: {mean_y_dataval_terz}")
print(f"La desviación estándar de y_dataval_terz es: {std_y_dataval_terz}")
print("")

# Calcular la media y desviación estándar para y_epri
mean_y_dataval_epri = y_dataval_epri.mean()
std_y_dataval_epri = y_dataval_epri.std()

print(f"La media de y_dataval_epri es: {mean_y_dataval_epri}")
print(f"La desviación estándar de y_dataval_epri es: {std_y_dataval_epri}")

In [ ]:
#@title Evaluación de modelos - Modelo ElasticNet

#Valores de predicción Terzija/Koglin de cada conjunto
y_pred_train_terzija_elastic_net = modelo_terzija_elastic_net.predict(x_train_terz)
y_pred_test_terzija_elastic_net = modelo_terzija_elastic_net.predict(x_test_terz)
y_pred_val_terzija_elastic_net = modelo_terzija_elastic_net.predict(x_dataval_terz)

#Valores de predicción EPRI de cada conjunto
y_pred_train_epri_elastic_net = modelo_epri_elastic_net.predict(x_train_epri)
y_pred_test_epri_elastic_net = modelo_epri_elastic_net.predict(x_test_epri)
y_pred_val_epri_elastic_net = modelo_epri_elastic_net.predict(x_dataval_epri)

# Obtener las métricas de evaluación para Terzija/Koglin - ElasticNet
print("Modelo ElasticNet - Metodología Terzija/Koglin")
mae_train_terz_elastic_net, rmse_train_terz_elastic_net, r2_train_terz_elastic_net, mae_test_terz_elastic_net, rmse_test_terz_elastic_net, r2_test_terz_elastic_net, mae_val_terz_elastic_net, rmse_val_terz_elastic_net, r2_val_terz_elastic_net = evaluar_modelo(y_train_terz, y_pred_train_terzija_elastic_net, y_test_terz, y_pred_test_terzija_elastic_net, y_dataval_terz, y_pred_val_terzija_elastic_net)

# Obtener las métricas de evaluación para EPRI - ElasticNet
print("Modelo ElasticNet - Metodología EPRI")
mae_train_epri_elastic_net, rmse_train_epri_elastic_net, r2_train_epri_elastic_net, mae_test_epri_elastic_net, rmse_test_epri_elastic_net, r2_test_epri_elastic_net, mae_val_epri_elastic_net, rmse_val_epri_elastic_net, r2_val_epri_elastic_net = evaluar_modelo(y_train_epri, y_pred_train_epri_elastic_net, y_test_epri, y_pred_test_epri_elastic_net, y_dataval_epri, y_pred_val_epri_elastic_net)

# Datos para las gráficas - ElasticNet
conjuntos = ['Entrenamiento', 'Prueba', 'Validación']

#R2
r2_terzija_elastic_net = [r2_train_terz_elastic_net, r2_test_terz_elastic_net, r2_val_terz_elastic_net]
r2_epri_elastic_net = [r2_train_epri_elastic_net, r2_test_epri_elastic_net, r2_val_epri_elastic_net]

#MAE
mae_terzija_elastic_net = [mae_train_terz_elastic_net, mae_test_terz_elastic_net, mae_val_terz_elastic_net]
mae_epri_elastic_net = [mae_train_epri_elastic_net, mae_test_epri_elastic_net, mae_val_epri_elastic_net]

#RMSE
rmse_terzija_elastic_net = [rmse_train_terz_elastic_net, rmse_test_terz_elastic_net, rmse_val_terz_elastic_net]
rmse_epri_elastic_net = [rmse_train_epri_elastic_net, rmse_test_epri_elastic_net, rmse_val_epri_elastic_net]

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'

# Gráfica para R2 - ElasticNet
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, r2_terzija_elastic_net, width=0.4, label='R2 Terzija/Koglin', color='purple')
plt.bar(np.arange(len(conjuntos)) + 0.4, r2_epri_elastic_net, width=0.4, label='R2 EPRI', color='aquamarine')
plt.axhline(y=1, color='b', linestyle='--', label='R2 = 1')
plt.title('Comparación de R2 - ElasticNet')
plt.xlabel('Conjunto de datos')
plt.ylabel('R2')
plt.legend()
file_path = os.path.join(dir_path, f'R2_ElasticNet.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()

# Gráfica para MAE - ElasticNet
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, mae_terzija_elastic_net, width=0.4, label='MAE Terzija/Koglin', color='maroon')
plt.bar(np.arange(len(conjuntos)) + 0.4, mae_epri_elastic_net, width=0.4, label='MAE EPRI', color='skyblue')
plt.axhline(y=mean_y_terz, color='r', linestyle='--', label='Media Terzija/Koglin')
plt.axhline(y=mean_y_epri, color='g', linestyle='--', label='Media EPRI')
plt.title('Comparación de MAE - ElasticNet')
plt.xlabel('Conjunto de datos')
plt.ylabel('MAE')
plt.legend()
file_path = os.path.join(dir_path, f'MAE_ElasticNet.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()

# Gráfica para RMSE - ElasticNet
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, rmse_terzija_elastic_net, width=0.4, label='RMSE Terzija/Koglin', color='darkgreen')
plt.bar(np.arange(len(conjuntos)) + 0.4, rmse_epri_elastic_net, width=0.4, label='RMSE EPRI', color='khaki')
plt.axhline(y=mean_y_terz, color='r', linestyle='--', label='Media Terzija/Koglin')
plt.axhline(y=mean_y_epri, color='g', linestyle='--', label='Media EPRI')
plt.title('Comparación de RMSE - ElasticNet')
plt.xlabel('Conjunto de datos')
plt.ylabel('RMSE')
plt.legend()
file_path = os.path.join(dir_path, f'RMSE_ElasticNet.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()


In [ ]:
#@title Evaluación de modelos - Modelo Polynomial Grade 2

#Valores de predicción Terzija/Koglin de cada conjunto
y_pred_train_terzija_polinomial = modelo_terzija_polinomial.predict(x_train_terz)
y_pred_test_terzija_polinomial = modelo_terzija_polinomial.predict(x_test_terz)
y_pred_val_terzija_polinomial = modelo_terzija_polinomial.predict(x_dataval_terz)

#Valores de predicción EPRI de cada conjunto
y_pred_train_epri_polinomial = modelo_epri_polinomial.predict(x_train_epri)
y_pred_test_epri_polinomial = modelo_epri_polinomial.predict(x_test_epri)
y_pred_val_epri_polinomial = modelo_epri_polinomial.predict(x_dataval_epri)

# Obtener las métricas de evaluación para Terzija/Koglin - Polynomial Grade 2
print("Modelo Polynomial Grade 2 - Metodología Terzija/Koglin")
mae_train_terz_polinomial, rmse_train_terz_polinomial, r2_train_terz_polinomial, mae_test_terz_polinomial, rmse_test_terz_polinomial, r2_test_terz_polinomial, mae_val_terz_polinomial, rmse_val_terz_polinomial, r2_val_terz_polinomial = evaluar_modelo(y_train_terz, y_pred_train_terzija_polinomial, y_test_terz, y_pred_test_terzija_polinomial, y_dataval_terz, y_pred_val_terzija_polinomial)

# Obtener las métricas de evaluación para EPRI - Polynomial Grade 2
print("Modelo Polynomial Grade 2 - Metodología EPRI")
mae_train_epri_polinomial, rmse_train_epri_polinomial, r2_train_epri_polinomial, mae_test_epri_polinomial, rmse_test_epri_polinomial, r2_test_epri_polinomial, mae_val_epri_polinomial, rmse_val_epri_polinomial, r2_val_epri_polinomial = evaluar_modelo(y_train_epri, y_pred_train_epri_polinomial, y_test_epri, y_pred_test_epri_polinomial, y_dataval_epri, y_pred_val_epri_polinomial)

# Datos para las gráficas - Polynomial Grade 2
conjuntos = ['Entrenamiento', 'Prueba', 'Validación']

#R2
r2_terzija_polinomial = [r2_train_terz_polinomial, r2_test_terz_polinomial, r2_val_terz_polinomial]
r2_epri_polinomial = [r2_train_epri_polinomial, r2_test_epri_polinomial, r2_val_epri_polinomial]

#MAE
mae_terzija_polinomial = [mae_train_terz_polinomial, mae_test_terz_polinomial, mae_val_terz_polinomial]
mae_epri_polinomial = [mae_train_epri_polinomial, mae_test_epri_polinomial, mae_val_epri_polinomial]

#RMSE
rmse_terzija_polinomial = [rmse_train_terz_polinomial, rmse_test_terz_polinomial, rmse_val_terz_polinomial]
rmse_epri_polinomial = [rmse_train_epri_polinomial, rmse_test_epri_polinomial, rmse_val_epri_polinomial]

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'

#Gráfica para R2 - Polynomial Grade 2
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, r2_terzija_polinomial, width=0.4, label='R2 Terzija/Koglin', color='purple')
plt.bar(np.arange(len(conjuntos)) + 0.4, r2_epri_polinomial, width=0.4, label='R2 EPRI', color='aquamarine')
plt.axhline(y=1, color='b', linestyle='--', label='R2 = 1')
plt.title('Comparación de R2 - Polynomial Grade 2')
plt.xlabel('Conjunto de datos')
plt.ylabel('R2')
plt.legend()
file_path = os.path.join(dir_path, f'R2_Polynomial_Grade_2.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()

#Gráfica para MAE - Polynomial Grade 2
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, mae_terzija_polinomial, width=0.4, label='MAE Terzija/Koglin', color='maroon')
plt.bar(np.arange(len(conjuntos)) + 0.4, mae_epri_polinomial, width=0.4, label='MAE EPRI', color='skyblue')
plt.axhline(y=mean_y_terz, color='r', linestyle='--', label='Media Terzija/Koglin')
plt.axhline(y=mean_y_epri, color='g', linestyle='--', label='Media EPRI')
plt.title('Comparación de MAE - Polynomial Grade 2')
plt.xlabel('Conjunto de datos')
plt.ylabel('MAE')
plt.legend()
file_path = os.path.join(dir_path, f'MAE_Polynomial_Grade_2.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()

#Gráfica para RMSE - Polynomial Grade 2
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, rmse_terzija_polinomial, width=0.4, label='RMSE Terzija/Koglin', color='darkgreen')
plt.bar(np.arange(len(conjuntos)) + 0.4, rmse_epri_polinomial, width=0.4, label='RMSE EPRI', color='khaki')
plt.axhline(y=mean_y_terz, color='r', linestyle='--', label='Media Terzija/Koglin')
plt.axhline(y=mean_y_epri, color='g', linestyle='--', label='Media EPRI')
plt.title('Comparación de RMSE - Polynomial Grade 2')
plt.xlabel('Conjunto de datos')
plt.ylabel('RMSE')
plt.legend()
file_path = os.path.join(dir_path, f'RMSE_Polynomial_Grade_2.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()


In [ ]:
#@title Evaluación de modelos - Modelo Neural Network

#Valores de predicción Terzija/Koglin de cada conjunto
y_pred_train_terzija_red_neuronal = modelo_terzija_red_neuronal.predict(x_train_terz)
y_pred_test_terzija_red_neuronal = modelo_terzija_red_neuronal.predict(x_test_terz)
y_pred_val_terzija_red_neuronal = modelo_terzija_red_neuronal.predict(x_dataval_terz)

#Valores de predicción EPRI de cada conjunto
y_pred_train_epri_red_neuronal = modelo_epri_red_neuronal.predict(x_train_epri)
y_pred_test_epri_red_neuronal = modelo_epri_red_neuronal.predict(x_test_epri)
y_pred_val_epri_red_neuronal = modelo_epri_red_neuronal.predict(x_dataval_epri)

#Obtener las métricas de evaluación para Terzija/Koglin - Neural Network
print("Modelo Neural Network - Metodología Terzija/Koglin")
mae_train_terz_red_neuronal, rmse_train_terz_red_neuronal, r2_train_terz_red_neuronal, mae_test_terz_red_neuronal, rmse_test_terz_red_neuronal, r2_test_terz_red_neuronal, mae_val_terz_red_neuronal, rmse_val_terz_red_neuronal, r2_val_terz_red_neuronal = evaluar_modelo(y_train_terz, y_pred_train_terzija_red_neuronal, y_test_terz, y_pred_test_terzija_red_neuronal, y_dataval_terz, y_pred_val_terzija_red_neuronal)

#Obtener las métricas de evaluación para Terzija/Koglin - Neural Network
print("Modelo Neural Network - Metodología EPRI")
mae_train_epri_red_neuronal, rmse_train_epri_red_neuronal, r2_train_epri_red_neuronal, mae_test_epri_red_neuronal, rmse_test_epri_red_neuronal, r2_test_epri_red_neuronal, mae_val_epri_red_neuronal, rmse_val_epri_red_neuronal, r2_val_epri_red_neuronal = evaluar_modelo(y_train_epri, y_pred_train_epri_red_neuronal, y_test_epri, y_pred_test_epri_red_neuronal, y_dataval_epri, y_pred_val_epri_red_neuronal)

# Datos para las gráficas - Neural Network
conjuntos = ['Entrenamiento', 'Prueba', 'Validación']

#R2
r2_terzija_red_neuronal = [r2_train_terz_red_neuronal, r2_test_terz_red_neuronal, r2_val_terz_red_neuronal]
r2_epri_red_neuronal = [r2_train_epri_red_neuronal, r2_test_epri_red_neuronal, r2_val_epri_red_neuronal]

#MAE
mae_terzija_red_neuronal = [mae_train_terz_red_neuronal, mae_test_terz_red_neuronal, mae_val_terz_red_neuronal]
mae_epri_red_neuronal = [mae_train_epri_red_neuronal, mae_test_epri_red_neuronal, mae_val_epri_red_neuronal]

#RMSE
rmse_terzija_red_neuronal = [rmse_train_terz_red_neuronal, rmse_test_terz_red_neuronal, rmse_val_terz_red_neuronal]
rmse_epri_red_neuronal = [rmse_train_epri_red_neuronal, rmse_test_epri_red_neuronal, rmse_val_epri_red_neuronal]

# Definir la ruta de guardado
dir_path = '15_kV_Arc_Flash/Archivos/Imagenes'

#Gráfica para R2 - Neural Network
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, r2_terzija_red_neuronal, width=0.4, label='R2 Terzija/Koglin', color='purple')
plt.bar(np.arange(len(conjuntos)) + 0.4, r2_epri_red_neuronal, width=0.4, label='R2 EPRI', color='aquamarine')
plt.axhline(y=1, color='b', linestyle='--', label='R2 = 1')
plt.title('Comparación de R2 - Neural Network')
plt.xlabel('Conjunto de datos')
plt.ylabel('R2')
plt.legend()
file_path = os.path.join(dir_path, f'R2_Neural_Network.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()

#Gráfica para MAE - Neural Network
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, mae_terzija_red_neuronal, width=0.4, label='MAE Terzija/Koglin', color='maroon')
plt.bar(np.arange(len(conjuntos)) + 0.4, mae_epri_red_neuronal, width=0.4, label='MAE EPRI', color='skyblue')
plt.axhline(y=mean_y_terz, color='r', linestyle='--', label='Media Terzija/Koglin')
plt.axhline(y=mean_y_epri, color='g', linestyle='--', label='Media EPRI')
plt.title('Comparación de MAE - Neural Network')
plt.xlabel('Conjunto de datos')
plt.ylabel('MAE')
plt.legend()
file_path = os.path.join(dir_path, f'MAE_Neural_Network.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()

#Gráfica para RMSE - Neural Network
plt.figure(figsize=(8, 6))
plt.bar(conjuntos, rmse_terzija_red_neuronal, width=0.4, label='RMSE Terzija/Koglin', color='darkgreen')
plt.bar(np.arange(len(conjuntos)) + 0.4, rmse_epri_red_neuronal, width=0.4, label='RMSE EPRI', color='khaki')
plt.axhline(y=mean_y_terz, color='r', linestyle='--', label='Media Terzija/Koglin')
plt.axhline(y=mean_y_epri, color='g', linestyle='--', label='Media EPRI')
plt.title('Comparación de RMSE - Neural Network')
plt.xlabel('Conjunto de datos')
plt.ylabel('RMSE')
plt.legend()
file_path = os.path.join(dir_path, f'RMSE_Neural_Network.png')
plt.savefig(file_path, bbox_inches='tight', pad_inches=0.1) # Guardar la figura
plt.show()


In [ ]:
#@title Guardar Modelos de regresión

# # Crear la carpeta 'Modelos' si no existe
# ruta_modelo = os.path.join('15_kV_Arc_Flash', 'Archivos', 'Modelos')
# if not os.path.exists(ruta_modelo):
#     os.makedirs(ruta_modelo)

# # Guardar los modelos
# joblib.dump(modelo_terzija_elastic_net, os.path.join(ruta_modelo, 'modelo_elastic_net_terzija.pkl'))
# joblib.dump(modelo_terzija_polinomial, os.path.join(ruta_modelo, 'modelo_polinomial_terzija.pkl'))
# joblib.dump(modelo_terzija_red_neuronal, os.path.join(ruta_modelo, 'modelo_red_neuronal_terzija.pkl'))
# joblib.dump(modelo_epri_elastic_net, os.path.join(ruta_modelo, 'modelo_elastic_net_epri.pkl'))
# joblib.dump(modelo_epri_polinomial, os.path.join(ruta_modelo, 'modelo_polinomial_epri.pkl'))
# joblib.dump(modelo_epri_red_neuronal, os.path.join(ruta_modelo, 'modelo_red_neuronal_epri.pkl'))

In [ ]:
#@title Prueba carga de modelos y su funcionamiento

# modelo_terzija_elastic_net = joblib.load(os.path.join(ruta_modelo, 'modelo_elastic_net_terzija.pkl'))
# modelo_terzija_polinomial = joblib.load(os.path.join(ruta_modelo, 'modelo_polinomial_terzija.pkl'))
# modelo_terzija_red_neuronal = joblib.load(os.path.join(ruta_modelo, 'modelo_red_neuronal_terzija.pkl'))
# modelo_epri_elastic_net = joblib.load(os.path.join(ruta_modelo, 'modelo_elastic_net_epri.pkl'))
# modelo_epri_polinomial = joblib.load(os.path.join(ruta_modelo, 'modelo_polinomial_epri.pkl'))
# modelo_epri_red_neuronal = joblib.load(os.path.join(ruta_modelo, 'modelo_red_neuronal_epri.pkl'))

# y_pred_train_terzija_elastic_net = modelo_terzija_elastic_net.predict(x_train_terz)
# y_pred_test_terzija_elastic_net = modelo_terzija_elastic_net.predict(x_test_terz)
# y_pred_val_terzija_elastic_net = modelo_terzija_elastic_net.predict(x_dataval_terz)

# print("Modelo ElasticNet - Metodología Terzija/Koglin")
# evaluar_modelo(y_train_terz, y_pred_train_terzija_elastic_net, y_test_terz, y_pred_test_terzija_elastic_net, y_dataval_terz, y_pred_val_terzija_elastic_net)

# y_pred_train_epri_elastic_net = modelo_epri_elastic_net.predict(x_train_epri)
# y_pred_test_epri_elastic_net = modelo_epri_elastic_net.predict(x_test_epri)
# y_pred_val_epri_elastic_net = modelo_epri_elastic_net.predict(x_dataval_epri)

# print("Modelo ElasticNet - Metodología EPRI")
# evaluar_modelo(y_train_epri, y_pred_train_epri_elastic_net, y_test_epri, y_pred_test_epri_elastic_net, y_dataval_epri, y_pred_val_epri_elastic_net)

# y_pred_train_terzija_polinomial = modelo_terzija_polinomial.predict(x_train_terz)
# y_pred_test_terzija_polinomial = modelo_terzija_polinomial.predict(x_test_terz)
# y_pred_val_terzija_polinomial = modelo_terzija_polinomial.predict(x_dataval_terz)

# print("Modelo Polynomial Grade 2 - Metodología Terzija/Koglin")
# evaluar_modelo(y_train_terz, y_pred_train_terzija_polinomial, y_test_terz, y_pred_test_terzija_polinomial, y_dataval_terz, y_pred_val_terzija_polinomial)

# y_pred_train_epri_polinomial = modelo_epri_polinomial.predict(x_train_epri)
# y_pred_test_epri_polinomial = modelo_epri_polinomial.predict(x_test_epri)
# y_pred_val_epri_polinomial = modelo_epri_polinomial.predict(x_dataval_epri)

# print("Modelo Polynomial Grade 2 - Metodología EPRI")
# evaluar_modelo(y_train_epri, y_pred_train_epri_polinomial, y_test_epri, y_pred_test_epri_polinomial, y_dataval_epri, y_pred_val_epri_polinomial)

# print("Modelo Polynomial Grade 2 - Metodología EPRI")
# evaluar_modelo(y_train_epri, y_pred_train_epri_polinomial, y_test_epri, y_pred_test_epri_polinomial, y_dataval_epri, y_pred_val_epri_polinomial)

# y_pred_train_terzija_red_neuronal = modelo_terzija_red_neuronal.predict(x_train_terz)
# y_pred_test_terzija_red_neuronal = modelo_terzija_red_neuronal.predict(x_test_terz)
# y_pred_val_terzija_red_neuronal = modelo_terzija_red_neuronal.predict(x_dataval_terz)

# print("Modelo Neural Network - Metodología Terzija/Koglin")
# evaluar_modelo(y_train_terz, y_pred_train_terzija_red_neuronal, y_test_terz, y_pred_test_terzija_red_neuronal, y_dataval_terz, y_pred_val_terzija_red_neuronal)

# y_pred_train_epri_red_neuronal = modelo_epri_red_neuronal.predict(x_train_epri)
# y_pred_test_epri_red_neuronal = modelo_epri_red_neuronal.predict(x_test_epri)
# y_pred_val_epri_red_neuronal = modelo_epri_red_neuronal.predict(x_dataval_epri)

# print("Modelo Neural Network - Metodología EPRI")
# evaluar_modelo(y_train_epri, y_pred_train_epri_red_neuronal, y_test_epri, y_pred_test_epri_red_neuronal, y_dataval_epri, y_pred_val_epri_red_neuronal)

#### Considerando los resultados obtenidos se selecciona el modelo de Redes Neuronales como Modelo de Ajuste de Energía Incidente (MAEI)

#Validación del MAEI

In [ ]:
#@title Cargar Modelo de Ajuste de Energía Incidente
MAEI_Terzija = joblib.load(os.path.join(ruta_modelo, 'modelo_red_neuronal_terzija_original.pkl'))
MAEI_EPRI = joblib.load(os.path.join(ruta_modelo, 'modelo_red_neuronal_epri_original.pkl'))

##Nivel de tensión menor a 15 kV

##Nivel de tensión mayor a 15 kV

###Pruebas con cálculos de literatura

En el artículo High Voltage Arc Flash Assessment and Applications, realizado por Albert Marroquin, Abdur Rehman y Ali Madani, se comparan valores de energía incidente en encerramientos a niveles de tensión de 5 kV, 15 kV, 25 kV y 35 kV. Los resultados se presentan en la Figura 6 del articulo y se extraen para ser objetivo de análisis.

In [ ]:
#@title Resultados Fig. 6
#Se crea un dataframe con los resultados de la Figura 6 del artículo High Voltage Arc Flash Assessment and Applications

WD_inch = 36; "[in]" #Distancia de trabajo
Tarc = 0.200; "[s]" #Tiempo de despeje de arco
Icc = 10; "[kA]" #Corriente de cortocircuito

# Crear el DataFrame
data = {
    "Nivel de tensión (kV)": [5, 15, 25, 35],
    "Gap (in)": [4, 6, 9, 12],
    "Gap (mm)": [101.6, 152.4, 228.6, 304.8],
    "IEEE 1584 2002 (cal/cm²)": [2.1, 2.5, 3.0, 3.7],
    "EPRI paper (cal/cm²)": [3.5, 4.1, 6.0, 7.2],
    "Terzija/Koglin paper (cal/cm²)": [3.6, 3.7, 4.8, 5.9],
    "ArcPro (cal/cm²)": [2.8, 3.0, 4.3, 5.9]
}

df = pd.DataFrame(data)

# Mostrar el DataFrame
print(df)

In [ ]:
#@title Cálculo con metodología Ralph Lee

# Calcular la energía incidente E para cada nivel de tensión
df["Ralph Lee (cal/cm²)"] = (793 * Icc * df["Nivel de tensión (kV)"] * Tarc) / (WD_inch ** 2)

# Mostrar el DataFrame con los nuevos cálculos
print(df)

Para alimentar el MAEI se debe tener en cuenta el siguiente orden para el ingreso de variables:

1. Tensión nominal (V) [kV]
2. Corriente de cortocircuito (Ibf) [kA]
3. Distancia de Trabajo (D) [mm]
4. Gap (G) [mm]
5. Configuración de electrodos (EC). Considerar: HCB -> 0; VCB -> 1; VCBB -> 2


In [ ]:
#@title Cálculo con Metodología EPRI Open Air
#Consideraciones:
#Este método utiliza como bases teóricas los estudios presentados en el Technical Report de EPRI “Arc Flash Issues in Transmission and Substation Environments”

#Suponemos la corriente de arco igual a la corriente de cortocicuito
Iarc = Icc

#El factor de ajuste estadístico se establece así:
sigma = 0.196 #Desviación estándar
nst = 3 #Factor de multiplicación estadístico. Se selecciona de 3 para obtener resultados más conservativos con la metodología EPRI
kst = 1 + nst * sigma #Factor de ajuste estadístico

#Distancia de trabajo a mm
WD = WD_inch * 25.4; "[mm]"

df_epri = df.copy()
# Columnas a conservar
columnas_a_conservar = ["Nivel de tensión (kV)", "Gap (mm)"]
# Elimina las columnas que no están en la lista 'columnas_a_conservar'
df_epri = df_epri[columnas_a_conservar]


# Crea nuevas columnas para los resultados
df_epri['Eave (kV_pico/m)'] = np.nan
df_epri['Earc EPRI aire libre (cal/cm²)'] = np.nan
df_epri['k_3f_enc HCB'] = np.nan #k_3f_enc HCB corresponde al factor de ajuste trifásico en encerramientos para un EC = HCB
df_epri['k_3f_enc VCB'] = np.nan #k_3f_enc VCB corresponde al factor de ajuste trifásico en encerramientos para un EC = VCB
df_epri['k_3f_enc VCBB'] = np.nan #k_3f_enc VCBB corresponde al factor de ajuste trifásico en encerramientos para un EC = VCBB
df_epri['Earc EPRI encerramiento HCB (cal/cm²)'] = np.nan
df_epri['Earc EPRI encerramiento VCB (cal/cm²)'] = np.nan
df_epri['Earc EPRI encerramiento VCBB (cal/cm²)'] = np.nan


# Itera sobre las filas del dataframe
for index, row in df_epri.iterrows():
    # Accede a las variables necesarias
    gap = row['Gap (mm)']
    V = row['Nivel de tensión (kV)']
    gap_m = gap * 0.001; "[m]"
    WD_m = WD * 0.001; "[m]"

    # Calcula Eave
    Eave = (0.0000112*(gap_m**(-8))) + 1.19 + (0.0069*(gap_m**(-1.239)) - 0.0126) * Iarc
    # Calcula Earc_epri_oa
    Earc_epri_oa = (6.7 * Eave * Iarc * (gap_m**0.58) * (3.281*WD_m)**(-1.58*(gap_m**(-0.152)))) * Tarc * kst
    #Factor de ajuste con MAEI
    k_3f_enc_HCB = MAEI_EPRI(V, Icc, WD, gap, 0)
    k_3f_enc_VCB = MAEI_EPRI(V, Icc, WD, gap, 1)
    k_3f_enc_VCBB = MAEI_EPRI(V, Icc, WD, gap, 2)
    #Energía incidente en encerramientos
    Earc_epri_enc_HCB = Earc_epri_oa * k_3f_enc_HCB
    Earc_epri_enc_VCB = Earc_epri_oa * k_3f_enc_VCB
    Earc_epri_enc_VCBB = Earc_epri_oa * k_3f_enc_VCBB

    # Guarda los resultados en las nuevas columnas
    df_epri.loc[index, 'Eave (kV_pico/m)'] = Eave
    df_epri.loc[index, 'Earc EPRI aire libre (cal/cm²)'] = Earc_epri_oa
    df_epri.loc[index, 'k_3f_enc HCB'] = k_3f_enc_HCB
    df_epri.loc[index, 'k_3f_enc VCB'] = k_3f_enc_VCB
    df_epri.loc[index, 'k_3f_enc VCBB'] = k_3f_enc_VCBB
    df_epri.loc[index, 'Earc EPRI encerramiento HCB (cal/cm²)'] = Earc_epri_enc_HCB
    df_epri.loc[index, 'Earc EPRI encerramiento VCB (cal/cm²)'] = Earc_epri_enc_VCB
    df_epri.loc[index, 'Earc EPRI encerramiento VCBB (cal/cm²)'] = Earc_epri_enc_VCBB


df_epri